In [ ]:
# @title




# # ===============================
# #   PART 0 — Install libraries
# # ===============================
# !pip install numpy pandas scikit-learn tensorflow matplotlib seaborn

# # ===============================
# #   PART 1 — Load NSL-KDD
# # ===============================
# import pandas as pd
# import numpy as np
# import tensorflow as tf
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score, classification_report
# from tensorflow.keras import layers, models
# # URLs of NSL-KDD files
# train_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"
# test_url  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt"

# cols = [
#     'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent',
#     'hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root',
#     'num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login',
#     'is_guest_login','count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
#     'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
#     'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
#     'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate',
#     'dst_host_srv_rerror_rate','label', 'difficulty_score' # Added 'difficulty_score' to account for all 43 columns
# ]

# train = pd.read_csv(train_url, names=cols)
# test  = pd.read_csv(test_url, names=cols)

# # Drop the difficulty_score column as it's not a feature for the model
# train = train.drop(columns=['difficulty_score'])
# test = test.drop(columns=['difficulty_score'])

# # Convert labels to binary
# train['label'] = train['label'].apply(lambda x: 0 if x == 'normal' else 1)
# test['label']  = test['label'].apply(lambda x: 0 if x == 'normal' else 1)

# # One-hot encode categorical
# cat = ['protocol_type','service','flag']
# train = pd.get_dummies(train, columns=cat)
# test = pd.get_dummies(test, columns=cat)

# # Align columns so train and test have the same set of features
# train_encoded, test_encoded = train.align(test, join='left', axis=1, fill_value=0)

# # Now both DataFrames have identical columns
# print("Train shape:", train_encoded.shape)
# print("Test shape:", test_encoded.shape)

# # test all numeric or not
# #print(train_encoded.select_dtypes(include=['object']).head())
# #print(test_encoded.select_dtypes(include=['object']).head())

# # divide data to xtrain and ytrain
# X_train = train_encoded.drop('label', axis=1).astype(float)
# y_train = train_encoded['label']

# X_test = test_encoded.drop('label', axis=1).astype(float)
# y_test = test_encoded['label']

# # Now both DataFrames have identical columns
# print("Train shape:", train_encoded.shape)
# print("Test shape:", test_encoded.shape)

# # ===============================
# #   PART 2 — Build IDS model (MLP)
# # ===============================

# # def build_ids(input_dim):
# #     model = models.Sequential([
# #         layers.Dense(64, activation='relu', input_shape=(input_dim,)),
# #         layers.Dense(32, activation='relu'),
# #         layers.Dense(1, activation='sigmoid')
# #     ])
# #     model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# #     return model

# # ids = build_ids(X_train.shape[1])
# # ids.fit(X_train, y_train, epochs=5, batch_size=256, validation_split=0.1)
# # loss, accuracy = ids.evaluate(X_test, y_test)
# # print(f"Test accuracy: {accuracy}")


# # ----------------------------
# # 1. Reshape input for LSTM
# # ----------------------------
# # LSTM expects: (batch, timesteps, features)
# # We treat each sample as 1 timestep with all features
# X_train_lstm = np.expand_dims(X_train, axis=1)   # shape becomes (N, 1, features)
# X_test_lstm  = np.expand_dims(X_test, axis=1)

# # ----------------------------
# # 2. Build LSTM-based IDS
# # ----------------------------
# def build_ids_lstm(input_dim,lr=0.001):
#     model = models.Sequential([
#         layers.LSTM(64, input_shape=(1, input_dim), return_sequences=False),
#         layers.Dense(32, activation='relu'),
#         layers.Dropout(0.3),
#         layers.Dense(1, activation='sigmoid')
#     ])

#     model.compile(  optimizer=tf.keras.optimizers.Adam(lr),
#                   loss='binary_crossentropy',
#                   metrics=['accuracy'])
#     return model

# ids = build_ids_lstm(X_train.shape[1])

# # ----------------------------
# # 3. Train the model
# # ----------------------------
# ids.fit(X_train_lstm, y_train,
#         epochs=50,
#         batch_size=256,
#         validation_split=0.1,
#         verbose=1)

# # ----------------------------
# # 4. Evaluate
# # ----------------------------
# loss, accuracy = ids.evaluate(X_test_lstm, y_test)
# print(f"Test accuracy: {accuracy}")

# # ===============================
# #   PART ----- —  an other way to Generate Adversarial Examples (FGSM) using problem space
# # ===============================
# # ===============================
# # Dhole Algorithm (2025)
# # ===============================

# def fitness_function(mask):
#     """
#     mask: binary vector (0/1) length = number of features
#     """
#     # avoid empty feature set
#     if np.sum(mask) == 0:
#         return np.inf

#     selected_idx = np.where(mask == 1)[0]

#     X_sel = X_train.iloc[:, selected_idx]
#     X_val_sel = X_test.iloc[:, selected_idx]

#     # simple IDS model (fast evaluation)
#     model = LogisticRegression(max_iter=200)
#     model.fit(X_sel, y_train)

#     y_pred = model.predict(X_val_sel)
#     acc = accuracy_score(y_test, y_pred)

#     # penalty for using too many features
#     penalty = 0.01 * np.sum(mask)

#     return -(acc - penalty)   # minimization


# import numpy as np

# def dhole_opt(N=20, T=5):
#     dim = X_train.shape[1]

#     # initialize population (binary masks)
#     X = np.random.randint(0, 2, size=(N, dim))

#     fitness = np.array([fitness_function(ind) for ind in X])

#     best_idx = np.argmin(fitness)
#     prey_global = X[best_idx].copy()
#     best_fit = fitness[best_idx]

#     curve = []

#     for t in range(T):
#         C = 1 - t / T

#         for i in range(N):
#             Xnew = X[i].copy()

#             if np.random.rand() < 0.5:
#                 # exploration
#                 j = np.random.randint(dim)
#                 Xnew[j] = 1 - Xnew[j]
#             else:
#                 # exploitation (move toward best)
#                 diff = prey_global ^ X[i]
#                 flip = np.random.rand(dim) < (C * diff)
#                 Xnew[flip] = prey_global[flip]

#             new_fit = fitness_function(Xnew)

#             if new_fit < fitness[i]:
#                 X[i] = Xnew
#                 fitness[i] = new_fit

#                 if new_fit < best_fit:
#                     best_fit = new_fit
#                     prey_global = Xnew.copy()

#         curve.append(best_fit)

#     return prey_global, best_fit, curve


# best_mask, best_score, curve = dhole_opt()

# selected_features = X_train.columns[best_mask == 1]

# print("Selected features:", list(selected_features))
# print("Number of features:", len(selected_features))


# model_opt = build_ids_lstm(len(selected_features))
# model_opt.fit(
#     X_train.iloc[:, selected_features], # Corrected indexing
#     y_train,
#     epochs=50,
#     batch_size=265
# )

# opt_acc = model_opt.evaluate(
#     X_test.iloc[:, selected_features], y_test # Corrected indexing
# )[1]

# print("Optimized Accuracy:", opt_acc)


# # ===============================
# #   PART 3 — Generate Adversarial Examples (FGSM) using feature space
# # ===============================

# # FGSM attack
# # def fgsm_attack(model, x, y, eps=0.1):
# #     x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
# #     y_tensor = tf.convert_to_tensor(y, dtype=tf.float32)

# #     # Reshape y_tensor to match the output shape of the model (batch_size, 1)
# #     y_tensor = tf.expand_dims(y_tensor, axis=-1)

# #     with tf.GradientTape() as tape:
# #         tape.watch(x_tensor)
# #         pred = model(x_tensor)
# #         loss = tf.keras.losses.binary_crossentropy(y_tensor, pred)

# #     grad = tape.gradient(loss, x_tensor)
# #     ae = x_tensor + eps * tf.sign(grad)
# #     return np.clip(ae.numpy(), 0, 1)

# # # Generate 5k adversarial samples
# # X_adv = fgsm_attack(ids, X_train[:5000], y_train[:5000])
# # y_adv = y_train[:5000].copy()
# # #show which features affected by FGSM
# # diff = X_adv - X_train[:5000]
# # changed_features = (diff != 0).sum(axis=0)
# # print(changed_features)

# # ===============================
# #   PART 3 — Generate Adversarial Examples (FGSM) using problem space
# # ===============================
# # ========== 1) Features allowed to be modified ==========
# # modifiable_features = [
# #     'duration','src_bytes','srv_count',
# #     'count','dst_host_count','dst_host_srv_count'
# # ]

# # p = 0.75 # = 5% max change per feature (problem space)

# # # copy of data before encoding
# # raw_test = pd.read_csv(test_url, names=cols)
# # raw_test = raw_test.drop(columns=['difficulty_score'])
# # raw_test['label'] = raw_test['label'].apply(lambda x: 0 if x == 'normal' else 1)

# # # ========== 2) FGSM on encoded space but apply to raw features ==========
# # def problem_space_fgsm(model, X_raw, X_encoded, y, eps=0.1):

# #     # convert int tensor
# #     X_raw = X_raw.copy()
# #     X = tf.convert_to_tensor(X_encoded, dtype=tf.float32)
# #     y = tf.convert_to_tensor(y, dtype=tf.float32)
# #     y = tf.expand_dims(y, axis=-1)

# #     with tf.GradientTape() as tape:
# #         tape.watch(X)
# #         pred = model(X)
# #         loss = tf.keras.losses.binary_crossentropy(y, pred)

# #     grad = tape.gradient(loss, X)

# #     # identify gradient of features
# #     grad_sign = np.sign(grad)

# #     # ========== 3) modify only modifiable features  ==========
# #     for f in modifiable_features:
# #         g = grad_sign[:, X_test.columns.get_loc(f)]

# #         # أقصى تغير مسموح به في problem-space p%
# #         delta = p * X_raw[f].abs()

# #         # القيمة الجديدة
# #         X_raw[f] = X_raw[f] + g * delta

# #         # ضمان عدم السالب
# #         X_raw[f] = np.clip(X_raw[f], 0, None)

# #     # ========== 4) apply One-hot ==========
# #     adv = pd.get_dummies(X_raw, columns=['protocol_type','service','flag'])

# #     # align مع train
# #     adv_encoded = adv.reindex(columns=test_encoded.columns, fill_value=0)

# #     # فصل X و y
# #     X_adv = adv_encoded.drop('label', axis=1).astype(float)
# #     y_adv = adv_encoded['label']

# #     return X_adv.values, y_adv.values


# # # # Generate 5k adversarial samples
# # X_raw_subset = raw_test.iloc[:5000].reset_index(drop=True)
# # X_encoded_subset = X_test.iloc[:5000].values
# # y_subset = y_test.iloc[:5000].values

# # # Generate adversarial samples for ALL test data
# # # X_raw_subset = raw_test.reset_index(drop=True)
# # # X_encoded_subset = X_test.values
# # # y_subset = y_test.values

# # X_adv, y_adv = problem_space_fgsm(ids, X_raw_subset, X_encoded_subset, y_subset, eps=0.1)

# # print("Generated adversarial examples (problem space):", X_adv.shape)

# # #show which features affected by FGSM
# # diff = X_adv - X_test[:5000]
# # changed_features = (diff != 0).sum(axis=0)
# # print(changed_features)



# # ===============================
# #   PART 3 —  an other way to Generate Adversarial Examples (FGSM) using problem space
# # ===============================
# # ===============================
# # FGSM EXACTLY LIKE MANDA (2024)
# # ===============================



# # -------- 1) Problem-space numeric features only --------
# numeric_features = [
#     'duration','src_bytes','count','srv_count','dst_host_count','dst_host_srv_count'
# ]

# # -------- 2) All categorical that must NOT change --------
# categorical_features = ['protocol_type','service','flag']
# # copy of data before encoding
# raw_test = pd.read_csv(test_url, names=cols)
# raw_test = raw_test.drop(columns=['difficulty_score'])
# raw_test['label'] = raw_test['label'].apply(lambda x: 0 if x == 'normal' else 1)
# X_raw_41 = raw_test.drop('label', axis=1)
# # X_encoded_121 = pd.get_dummies(X_raw_41, columns=categorical_features)
# # X_train_encoded_cols = X_encoded_121.columns

# def manda_fgsm(model, X_raw_unencoded, X_encoded_aligned, y_labels, eps=0.1, model_expected_cols=None,p=0.05):

#     # ---------- Step 1: FGSM in feature-space ----------
#     X = tf.convert_to_tensor(X_encoded_aligned, dtype=tf.float32)
#     X = tf.expand_dims(X, axis=1) # Add the timesteps dimension for LSTM
#     y_tensor = tf.convert_to_tensor(y_labels, dtype=tf.float32)
#     y_tensor = tf.expand_dims(y_tensor, axis=-1)

#     with tf.GradientTape() as tape:
#         tape.watch(X)
#         pred = model(X)
#         loss = tf.keras.losses.binary_crossentropy(y_tensor, pred)

#     grad = tape.gradient(loss, X).numpy()
#     grad_sign = np.sign(grad)

#     # Ensure model_expected_cols is provided
#     if model_expected_cols is None:
#         raise ValueError("model_expected_cols must be provided for problem-space FGSM")

#     # Nullify perturbations on categorical / non-diff features
#     # Iterate through categorical features and find their one-hot encoded columns in model_expected_cols
#     for cat in categorical_features:
#         # Find indices of one-hot encoded columns corresponding to the categorical feature
#         col_indices = [i for i, c in enumerate(model_expected_cols) if c.startswith(cat+'_')]
#         if col_indices: # Only modify if such columns exist
#             grad_sign[:, 0, col_indices] = 0 # Adjust index for 3D grad_sign

#     # ---------- Step 2: Map back to problem-space ----------
#     X_raw_adv = X_raw_unencoded.copy()

#     for f in numeric_features:
#         # Check if the numeric feature exists in the model's expected columns
#         if f in model_expected_cols:
#             # Apply perturbation to the raw feature based on the gradient of its encoded counterpart
#             # The gradient sign for 'f' is at col_index_in_encoded_aligned in grad_sign
#             idx = model_expected_cols.get_loc(f)
#             g = grad_sign[:, 0, idx]               # FGSM direction (+1 / -1) - Adjusted for 3D grad_sign

#             delta = p * X_raw_adv[f].abs()     # allowed change = p%

#             # apply modification
#             X_raw_adv[f] = X_raw_adv[f] +  (eps * g * delta)
#             # X_raw_adv[f] = X_raw_adv[f] + eps * grad_sign[:, idx] * np.abs(X_raw_unencoded[f])
#             X_raw_adv[f] = np.clip(X_raw_adv[f], 0, None) # Ensure non-negative

#     # Re-encode categorical features from the modified raw data
#     adv = pd.get_dummies(X_raw_adv, columns=categorical_features)

#     # Align with model's expected columns (X_train.columns)
#     X_adv_encoded = adv.reindex(columns=model_expected_cols, fill_value=0)

#     # Separate X and y (y labels remain unchanged, as it's an adversarial attack on X)
#     X_final = X_adv_encoded.astype(float).values # Removed .drop('label', axis=1)
#     y_final = y_labels # Labels remain the same

#     return X_final, y_final


# # # # Generate 5k adversarial samples
# X_raw_subset = X_raw_41.iloc[:200].reset_index(drop=True)
# X_encoded_subset = X_test.iloc[:200].values # Use the correctly aligned X_test data
# y_subset = y_test.iloc[:200].values

# X_adv, y_adv = manda_fgsm(model_opt,X_test.iloc[:, selected_features], X_encoded_subset, y_subset, eps=0.1, model_expected_cols=X_train.columns,p=0.075) # Pass X_train.columns for alignment # Corrected indexing

# print("Generated adversarial examples (problem space):", X_adv.shape)

# #show which features affected by FGSM
# diff = pd.DataFrame(X_adv, columns=X_train.columns) - X_test.iloc[:200] # Convert X_adv to DataFrame with correct columns for comparison
# changed_features = (diff != 0).sum(axis=0)
# print(changed_features)

# # ===============================
# #   PART 4 — Compute MANIFOLD SCORE
# # ===============================

# # PCA projection distance as manifold score
# from sklearn.decomposition import PCA
# from sklearn.metrics import pairwise_distances

# pca = PCA(n_components=10)
# pca.fit(X_train)

# def manifold_score(x):
#     proj = pca.inverse_transform(pca.transform(x))
#     return np.mean((x - proj)**2, axis=1)

# manifold_clean = manifold_score(X_train[:200])
# manifold_adv   = manifold_score(X_adv)


# # ===============================
# #   PART 5 — Compute DB SCORE (Deep Boundary)
# # ===============================

# def db_score(model, x):
#     # Reshape input for the LSTM model
#     x_reshaped = np.expand_dims(x, axis=1)
#     with tf.GradientTape() as tape:
#         x_t = tf.convert_to_tensor(x_reshaped, dtype=tf.float32)
#         tape.watch(x_t)
#         pred = model(x_t)
#     grad = tape.gradient(pred, x_t).numpy()
#     # The gradient will also be 3D (batch, 1, features), so squeeze the middle dimension
#     return np.mean(np.abs(grad[:, 0, :]), axis=1)

# db_clean = db_score(ids, X_train[:200])
# db_adv   = db_score(ids, X_adv)


# # ===============================
# #   PART 6 — Build MANDA dataset
# # ===============================

# S1 = np.concatenate([manifold_clean, manifold_adv])
# S2 = np.concatenate([db_clean, db_adv])
# Y  = np.concatenate([np.zeros_like(manifold_clean), np.ones_like(manifold_adv)])

# df_manda = pd.DataFrame({'manifold': S1, 'db': S2, 'label': Y})
# df_manda.head()
# #print(df_manda)

# # ===============================
# #   PART 7 — Train MANDA (Logistic Regression)
# # ===============================

# from sklearn.metrics import accuracy_score, roc_auc_score

# clf = LogisticRegression()
# clf.fit(df_manda[['manifold', 'db']], df_manda['label'])

# pred = clf.predict(df_manda[['manifold', 'db']])
# print("MANDA accuracy:", accuracy_score(df_manda['label'], pred))
# print("AUC:", roc_auc_score(df_manda['label'], pred))

# # ===============================
# #   PART 8 — Fixing FPR at 5% or 15%
# # ===============================

# from sklearn.metrics import roc_curve

# scores = clf.predict_proba(df_manda[['manifold', 'db']])[:,1]
# fpr, tpr, th = roc_curve(df_manda['label'], scores)

# def get_threshold(target_fpr):
#     idx = np.argmin(np.abs(fpr - target_fpr))
#     return th[idx]

# thr_5  = get_threshold(0.05)
# thr_15 = get_threshold(0.15)

# print("Threshold at 5% FPR:", thr_5)
# print("Threshold at 15% FPR:", thr_15)


In [ ]:
# ===============================
#   PART 0 — Install libraries
# ===============================
!pip install numpy pandas scikit-learn tensorflow matplotlib seaborn
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [ ]:

# ===============================
#   PART 1 — Load NSL-KDD
# ===============================
# URLs of NSL-KDD files
train_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"
test_url  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt"

cols = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent',
    'hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
    'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate',
    'dst_host_srv_rerror_rate','label', 'difficulty_score' # Added 'difficulty_score' to account for all 43 columns
]

train = pd.read_csv(train_url, names=cols)
test  = pd.read_csv(test_url, names=cols)

# Drop the difficulty_score column as it's not a feature for the model
train = train.drop(columns=['difficulty_score'])
test = test.drop(columns=['difficulty_score'])

# Convert labels to binary
train['label'] = train['label'].apply(lambda x: 0 if x == 'normal' else 1)
test['label']  = test['label'].apply(lambda x: 0 if x == 'normal' else 1)

# Catogrical features
cat = ['protocol_type','service','flag']

# # One-hot encode categorical
# train = pd.get_dummies(train, columns=cat)
# test = pd.get_dummies(test, columns=cat)

# label encoding
encoders = {}

for col in cat:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col]  = le.transform(test[col])
    encoders[col] = le

# Align columns so train and test have the same set of features
train_encoded, test_encoded = train.align(test, join='left', axis=1, fill_value=0)

# Now both DataFrames have identical columns
print("Train shape:", train_encoded.shape)
print("Test shape:", test_encoded.shape)

# test all numeric or not
#print(train_encoded.select_dtypes(include=['object']).head())
#print(test_encoded.select_dtypes(include=['object']).head())

# divide data to xtrain and ytrain
X_train = train_encoded.drop('label', axis=1).astype(float)
y_train = train_encoded['label']

X_test = test_encoded.drop('label', axis=1).astype(float)
y_test = test_encoded['label']

# Now both DataFrames have identical columns
print("Train shape:", train_encoded.shape)
print("Test shape:", test_encoded.shape)

Train shape: (125973, 42)
Test shape: (22544, 42)
Train shape: (125973, 42)
Test shape: (22544, 42)


In [ ]:


# ===============================
#   PART 2 — Build IDS model (LSTM)
# ===============================

# ----------------------------
# 1. Reshape input for LSTM
# ----------------------------
# LSTM expects: (batch, timesteps, features)
# We treat each sample as 1 timestep with all features
X_train_lstm = np.expand_dims(X_train, axis=1)   # shape becomes (N, 1, features)
X_test_lstm  = np.expand_dims(X_test, axis=1)

# Define timesteps for LSTM
timesteps_for_lstm = 5 # Consistent timesteps

#add function to change timestep >1
def make_lstm_sequences(X, y, timesteps=timesteps_for_lstm):
    X_seq, y_seq = [], []

    for i in range(len(X) - timesteps + 1):
        X_seq.append(X[i:i+timesteps])
        y_seq.append(y[i+timesteps-1])  # label آخر خطوة

    return np.array(X_seq), np.array(y_seq)

X_train_lstm, y_train_lstm = make_lstm_sequences(
    X_train.values, y_train.values, timesteps=5
)

X_test_lstm, y_test_lstm = make_lstm_sequences(
    X_test.values, y_test.values, timesteps=5
)

print(X_train_lstm.shape)  # (N-4, 5, features)
print(X_test_lstm.shape)



# ----------------------------
# 2. Build LSTM-based IDS
# ----------------------------

def build_ids_lstm(input_dim,lr=0.001):
    model = models.Sequential([
        layers.LSTM(128, input_shape=(timesteps_for_lstm, input_dim), return_sequences=True),
        layers.Dropout(0.3),

        layers.LSTM(64, return_sequences=False),
        layers.BatchNormalization(),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

ids = build_ids_lstm(X_train.shape[1])

# ----------------------------
# 3. Train the model
# ----------------------------
ids.fit(X_train_lstm, y_train_lstm,
        epochs=10,
        batch_size=256,
        validation_split=0.2,
        verbose=1)

# ----------------------------
# 4. Evaluate
# ----------------------------
loss, accuracy = ids.evaluate(X_test_lstm, y_test_lstm)
print("Test shape:", X_test_lstm.shape)
print(f"Test accuracy: {accuracy}")


(125969, 5, 41)
(22540, 5, 41)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 23s 48ms/step - accuracy: 0.8913 - loss: 0.2448 - val_accuracy: 0.9679 - val_loss: 0.0740
Epoch 2/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 20s 50ms/step - accuracy: 0.9670 - loss: 0.0833 - val_accuracy: 0.9669 - val_loss: 0.0659
Epoch 3/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - accuracy: 0.9698 - loss: 0.0745 - val_accuracy: 0.9749 - val_loss: 0.0568
Epoch 4/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.9748 - loss: 0.0659 - val_accuracy: 0.9807 - val_loss: 0.0562
Epoch 5/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.9752 - loss: 0.0655 - val_accuracy: 0.9831 - val_loss: 0.0537
Epoch 6/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.9771 - loss: 0.0605 - val_accuracy: 0.9817 - val_loss: 0.0497
Epoch 7/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 18s 46ms/step - accuracy: 0.9741 - loss: 0.0655 - val_accuracy: 0.9864 - val_loss: 0.0442
Epoch 8/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.9809 - loss: 0.0518 - 

In [ ]:
# ===== build new model on all features=====

# First extract numeric features indices
categorical_features = ['protocol_type','service','flag']
numeric_idx = [
    i for i, col in enumerate(X_train.columns)
    if not any(col.startswith(cat + '_') for cat in categorical_features)
]

####training and testing data for numeric features
X_train_numeric_values = X_train.values[:, numeric_idx]  # numpy array
X_test_numeric_values = X_test.values[:, numeric_idx]  # numpy array

# new model
ids_numeric = build_ids_lstm(len(numeric_idx))

#generate train data for lstm
X_train_numeric_lstm, y_train_numeric_lstm = make_lstm_sequences(
    X_train_numeric_values,
    y_train.values,
    timesteps=timesteps_for_lstm
)

#generate test data for lstm
X_test_numeric_lstm, y_test_numeric_lstm = make_lstm_sequences(
    X_test_numeric_values,
    y_test.values,
    timesteps=timesteps_for_lstm
)
#train new model on numeric features
ids_numeric.fit(
    X_train_numeric_lstm,
    y_train_numeric_lstm,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)
# ----------------------------
# 4. Evaluate
# ----------------------------
loss, accuracy = ids_numeric.evaluate(X_test_numeric_lstm, y_test_numeric_lstm)
print("Test shape:", X_test_numeric_lstm.shape)
print(f"Test accuracy: {accuracy}")


Epoch 1/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 32s 67ms/step - accuracy: 0.9083 - loss: 0.2099 - val_accuracy: 0.9648 - val_loss: 0.0775
Epoch 2/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 20s 50ms/step - accuracy: 0.9668 - loss: 0.0819 - val_accuracy: 0.9802 - val_loss: 0.0590
Epoch 3/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.9724 - loss: 0.0726 - val_accuracy: 0.9837 - val_loss: 0.0501
Epoch 4/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.9751 - loss: 0.0710 - val_accuracy: 0.9785 - val_loss: 0.0552
Epoch 5/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.9745 - loss: 0.0647 - val_accuracy: 0.9832 - val_loss: 0.0469
Epoch 6/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 22s 51ms/step - accuracy: 0.9749 - loss: 0.0619 - val_accuracy: 0.9805 - val_loss: 0.0465
Epoch 7/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 21s 52ms/step - accuracy: 0.9759 - loss: 0.0597 - val_accuracy: 0.9791 - val_loss: 0.0493
Epoch 8/10
394/394 ━━━━━━━━━━━━━━━━━━━━ 40s 51ms/step - accuracy: 0.9761 - loss: 0.0569 - 

In [ ]:
#perform attack on all numeric features


# ===============================
#   PART 3 —  an other way to Generate Adversarial Examples (FGSM) using problem space
# ===============================
# ===============================
# FGSM EXACTLY LIKE MANDA (2024)
# ===============================
# -------- 1) Problem-space all numeric features only --------
numeric_features = [
    col for col in X_test.columns
    if  col not in categorical_features
]

# -------- 2) All categorical that must NOT change --------
# copy of data before encoding
# raw_test = pd.read_csv(test_url, names=cols)
# raw_test = raw_test.drop(columns=['difficulty_score'])
# raw_test['label'] = raw_test['label'].apply(lambda x: 0 if x == 'normal' else 1)
# X_raw_41 = raw_test.drop('label', axis=1)
# X_encoded_121 = pd.get_dummies(X_raw_41, columns=categorical_features)
# X_train_encoded_cols = X_encoded_121.columns

def manda_fgsm(model, X_raw_unencoded_full, X_encoded_aligned_for_model, y_labels, eps=0.05, full_model_cols=None, subset_model_cols=None, p=0.05):

    # ---------- Step 1: FGSM in feature-space (on subset features) ----------
    X_model_input_tensor = tf.convert_to_tensor(X_encoded_aligned_for_model, dtype=tf.float32)
    X_model_input_tensor = tf.expand_dims(X_model_input_tensor, axis=1) # Add the timesteps dimension for LSTM
    y_tensor = tf.convert_to_tensor(y_labels, dtype=tf.float32)
    y_tensor = tf.expand_dims(y_tensor, axis=-1)

    with tf.GradientTape() as tape:
        tape.watch(X_model_input_tensor)
        pred = model(X_model_input_tensor)
        loss = tf.keras.losses.binary_crossentropy(y_tensor, pred)

    grad = tape.gradient(loss, X_model_input_tensor).numpy()
    grad_sign = np.sign(grad) # grad_sign will have shape (batch_size, 1, len(subset_model_cols))

    if full_model_cols is None or subset_model_cols is None:
        raise ValueError("full_model_cols and subset_model_cols must be provided for problem-space FGSM")

    # Nullify perturbations on categorical / non-diff features within the subset
    for cat in categorical_features:
        # Find indices of one-hot encoded columns corresponding to the categorical feature within subset_model_cols
        col_indices_in_subset = [i for i, c in enumerate(subset_model_cols) if c.startswith(cat+'_')]
        if col_indices_in_subset:
            grad_sign[:, 0, col_indices_in_subset] = 0 # This now correctly indexes into the subset grad_sign

    # ---------- Step 2: Map back to problem-space ----------
    X_raw_adv = X_raw_unencoded_full.copy() # Operate on the full raw DataFrame

    for f in numeric_features:
        # Check if the numeric feature exists in the subset_model_cols (because grad_sign is based on it)
        if f in subset_model_cols:
            idx_in_subset = subset_model_cols.get_loc(f)
            g = grad_sign[:, 0, idx_in_subset] # FGSM direction (+1 / -1)

            delta = p * X_raw_adv[f].abs()     # allowed change = p%

            # apply modification
            X_raw_adv[f] = X_raw_adv[f] +  (eps * g * delta)
            X_raw_adv[f] = np.clip(X_raw_adv[f], 0, None) # Ensure non-negative

    # Re-encode categorical features from the modified raw data
    adv = pd.get_dummies(X_raw_adv, columns=categorical_features)

    # Align with the full model's expected columns (X_train.columns) for the final output
    X_adv_encoded = adv.reindex(columns=full_model_cols, fill_value=0)

    # Separate X and y
    X_final = X_adv_encoded.astype(float).values
    y_final = y_labels

    return X_final, y_final


# # # Generate 5k adversarial samples (using 200 samples for consistency)
X_train_numeric_raw = X_train.iloc[:, numeric_idx].reset_index(drop=True)
# Filter X_test for selected features and consistent row count
X_train_numeric_encoded  = X_train.values[:, numeric_idx]
y_subset = y_train.values

# Diagnostic prints to verify shapes before calling manda_fgsm
print(f"Shape of X_raw_subset_full: {X_train_numeric_raw.shape}")
print(f"Shape of X_test_encoded_for_model: {X_train_numeric_encoded.shape}")
print(f"Shape of y_subset: {y_subset.shape}")

X_advtrain, y_advtrain = manda_fgsm(
    ids_numeric,
    X_train_numeric_raw,
    X_train_numeric_encoded,
    y_subset,
    eps=0.05,
    full_model_cols=X_train_numeric_raw.columns,
    subset_model_cols=X_train_numeric_raw.columns,
    p=0.05
)

print("Generated adversarial examples (problem space):", X_advtrain.shape)

#show which features affected by FGSM
diff = pd.DataFrame(X_advtrain, columns=X_train.columns) - X_test # Convert X_adv to DataFrame with correct columns for comparison
changed_features = (diff[numeric_features] != 0).sum(axis=0)
print(changed_features)



Shape of X_raw_subset_full: (125973, 41)
Shape of X_test_encoded_for_model: (125973, 41)
Shape of y_subset: (125973,)
Generated adversarial examples (problem space): (125973, 41)
duration                       108471
src_bytes                      123034
dst_bytes                      120975
land                           103438
wrong_fragment                 103719
urgent                         103440
hot                            104893
num_failed_logins              103930
logged_in                      118133
num_compromised                104044
root_shell                     103520
su_attempted                   103452
num_root                       103601
num_file_creations             103529
num_shells                     103457
num_access_files               103573
num_outbound_cmds              103429
is_host_login                  103440
is_guest_login                 104268
count                          125895
srv_count                      125897
serror_rate            

In [ ]:
#perform attack on all numeric features on test data


# ===============================
#   PART 3 —  an other way to Generate Adversarial Examples (FGSM) using problem space
# ===============================
# ===============================
# FGSM EXACTLY LIKE MANDA (2024)
# ===============================
# -------- 1) Problem-space all numeric features only --------
categorical_features = ['protocol_type','service','flag']
numeric_features = [
    col for col in X_test.columns
    if  col not in categorical_features
]

# -------- 2) All categorical that must NOT change --------
# copy of data before encoding
# raw_test = pd.read_csv(test_url, names=cols)
# raw_test = raw_test.drop(columns=['difficulty_score'])
# raw_test['label'] = raw_test['label'].apply(lambda x: 0 if x == 'normal' else 1)
# X_raw_41 = raw_test.drop('label', axis=1)
# X_encoded_121 = pd.get_dummies(X_raw_41, columns=categorical_features)
# X_train_encoded_cols = X_encoded_121.columns

def manda_fgsm(model, X_raw_unencoded_full, X_encoded_aligned_for_model, y_labels, eps=0.05, full_model_cols=None, subset_model_cols=None, p=0.05):

    # ---------- Step 1: FGSM in feature-space (on subset features) ----------
    X_model_input_tensor = tf.convert_to_tensor(X_encoded_aligned_for_model, dtype=tf.float32)
    X_model_input_tensor = tf.expand_dims(X_model_input_tensor, axis=1) # Add the timesteps dimension for LSTM
    y_tensor = tf.convert_to_tensor(y_labels, dtype=tf.float32)
    y_tensor = tf.expand_dims(y_tensor, axis=-1)

    with tf.GradientTape() as tape:
        tape.watch(X_model_input_tensor)
        pred = model(X_model_input_tensor)
        loss = tf.keras.losses.binary_crossentropy(y_tensor, pred)

    grad = tape.gradient(loss, X_model_input_tensor).numpy()
    grad_sign = np.sign(grad) # grad_sign will have shape (batch_size, 1, len(subset_model_cols))

    if full_model_cols is None or subset_model_cols is None:
        raise ValueError("full_model_cols and subset_model_cols must be provided for problem-space FGSM")

    # Nullify perturbations on categorical / non-diff features within the subset
    for cat in categorical_features:
        # Find indices of one-hot encoded columns corresponding to the categorical feature within subset_model_cols
        col_indices_in_subset = [i for i, c in enumerate(subset_model_cols) if c.startswith(cat+'_')]
        if col_indices_in_subset:
            grad_sign[:, 0, col_indices_in_subset] = 0 # This now correctly indexes into the subset grad_sign

    # ---------- Step 2: Map back to problem-space ----------
    X_raw_adv = X_raw_unencoded_full.copy() # Operate on the full raw DataFrame

    for f in numeric_features:
        # Check if the numeric feature exists in the subset_model_cols (because grad_sign is based on it)
        if f in subset_model_cols:
            idx_in_subset = subset_model_cols.get_loc(f)
            g = grad_sign[:, 0, idx_in_subset] # FGSM direction (+1 / -1)

            delta = p * X_raw_adv[f].abs()     # allowed change = p%

            # apply modification
            X_raw_adv[f] = X_raw_adv[f] +  (eps * g * delta)
            X_raw_adv[f] = np.clip(X_raw_adv[f], 0, None) # Ensure non-negative

    # Re-encode categorical features from the modified raw data
    adv = pd.get_dummies(X_raw_adv, columns=categorical_features)

    # Align with the full model's expected columns (X_train.columns) for the final output
    X_adv_encoded = adv.reindex(columns=full_model_cols, fill_value=0)

    # Separate X and y
    X_final = X_adv_encoded.astype(float).values
    y_final = y_labels

    return X_final, y_final


# # # Generate 5k adversarial samples (using 200 samples for consistency)
X_test_numeric_raw = X_test.iloc[:, numeric_idx].reset_index(drop=True)
# Filter X_test for selected features and consistent row count
X_test_numeric_encoded = X_test.values[:, numeric_idx]
y_subset = y_test.values

# Diagnostic prints to verify shapes before calling manda_fgsm
print(f"Shape of X_raw_subset_full: {X_test_numeric_raw.shape}")
print(f"Shape of X_test_encoded_for_model: {X_test_numeric_encoded.shape}")
print(f"Shape of y_subset: {y_subset.shape}")

X_advtest, y_advtest = manda_fgsm(
    ids_numeric,
    X_test_numeric_raw,
    X_test_numeric_encoded,
    y_subset,
    eps=0.05,
    full_model_cols=X_train_numeric_raw.columns,
    subset_model_cols=X_train_numeric_raw.columns,
    p=0.05
)

print("Generated adversarial examples (problem space):", X_advtest.shape)

#show which features affected by FGSM
diff = pd.DataFrame(X_advtest, columns=X_train.columns) - X_test # Convert X_adv to DataFrame with correct columns for comparison
changed_features = (diff[numeric_features] != 0).sum(axis=0)
print(changed_features)



Shape of X_raw_subset_full: (22544, 41)
Shape of X_test_encoded_for_model: (22544, 41)
Shape of y_subset: (22544,)
Generated adversarial examples (problem space): (22544, 41)
duration                        2949
src_bytes                      13692
dst_bytes                      12496
land                               7
wrong_fragment                   100
urgent                            10
hot                              635
num_failed_logins                477
logged_in                       9043
num_compromised                   48
root_shell                        49
su_attempted                       2
num_root                          41
num_file_creations                41
num_shells                        18
num_access_files                  69
num_outbound_cmds                  0
is_host_login                     11
is_guest_login                   641
count                          21314
srv_count                      21311
serror_rate                     3170
srv_serror_

In [ ]:
# training data after attack on all numeric feaures using LSTM
numeric_idx = [
    X_train.columns.get_loc(f)
    for f in X_train.columns
    if not any(f.startswith(cat + '_') for cat in categorical_features)
]
# clean
X_train_num = X_train.values[:, numeric_idx]
X_test_num  = X_test.values[:, numeric_idx] # X_test_num now has only numeric features

# adversarial (generated on numeric train features)
X_adv_numtrain = X_advtrain[:, numeric_idx] # X_adv_num now has only numeric features

# adversarial (generated on numeric test features)
X_adv_numtest = X_advtest[:, numeric_idx] # X_adv_num now has only numeric features

#merge clean and adversarial data together
X_train_adv = np.vstack([X_train_num, X_adv_numtrain])
y_train_adv = np.concatenate([y_train.values, y_advtrain])

# shuffle
idx = np.random.permutation(len(X_train_adv))
X_train_adv = X_train_adv[idx]
y_train_adv = y_train_adv[idx]


#add lstm dimesion in 5 time step for training
X_train_combined_lstm, y_train_combined_lstm = make_lstm_sequences(
    X_train_adv, y_train_adv, timesteps=timesteps_for_lstm
)

# Prepare clean test data with numeric features and 5 timesteps
X_test_numeric_lstm, y_test_numeric_lstm = make_lstm_sequences(
    X_test_num, y_test.values, timesteps=timesteps_for_lstm
)

# Prepare adversarial test data with numeric features and 5 timesteps
X_adv_numeric_lstm, y_adv_numeric_lstm = make_lstm_sequences(
    X_adv_numtest, y_advtest, timesteps=timesteps_for_lstm
)

##################################### type one ################################
#train LSTM  on training data and adversarial data together
# Train: Clean + FGSM
# Test : Adversarial
ids_robust  = build_ids_lstm(len(numeric_idx))
history = ids_robust .fit(
    X_train_combined_lstm,
    y_train_combined_lstm,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)
#accuracy on clean data
loss_clean, acc_adv_clean = ids_robust .evaluate(X_test_numeric_lstm, y_test_numeric_lstm)
print("Accuracy on clean test:", acc_clean)

#accuracy on adversarial data
loss_adv, acc_adv_adv = ids_robust .evaluate(X_adv_numeric_lstm, y_adv_numeric_lstm)
print("Accuracy on adversarial test:", acc_adv)


##############################################type two #############################
#train LSTM  on clean training data
# Train: Clean
# Test : Adversarial
#add lstm dimesion in 5 time step for training
# X_train_num_lstm,Y_train_num_lstm = make_lstm_sequences(
#     X_train_num, y_train, timesteps=timesteps_for_lstm
# )

# fgsmclean= build_ids_lstm(len(numeric_idx))
# history = fgsmclean.fit(
#     X_train_num_lstm,
#     Y_train_num_lstm,
#     epochs=50,
#     batch_size=256,
#     validation_split=0.2,
#     verbose=1
# )

# #accuracy on clean data
# loss_clean, acc_clean = fgsmclean.evaluate(X_test_numeric_lstm, y_test_numeric_lstm)
# print("Accuracy on clean test:", acc_clean)

# #accuracy on adversarial data
# loss_adv, acc_adv = fgsmclean.evaluate(X_adv_numeric_lstm, y_adv_numeric_lstm)
# print("Accuracy on adversarial test:", acc_adv)


Epoch 1/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 42s 48ms/step - accuracy: 0.9250 - loss: 0.1760 - val_accuracy: 0.9798 - val_loss: 0.0556
Epoch 2/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 38s 49ms/step - accuracy: 0.9719 - loss: 0.0737 - val_accuracy: 0.9738 - val_loss: 0.0569
Epoch 3/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 37s 47ms/step - accuracy: 0.9731 - loss: 0.0679 - val_accuracy: 0.9863 - val_loss: 0.0457
Epoch 4/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.9782 - loss: 0.0604 - val_accuracy: 0.9818 - val_loss: 0.0454
Epoch 5/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 38s 48ms/step - accuracy: 0.9770 - loss: 0.0608 - val_accuracy: 0.9882 - val_loss: 0.0404
Epoch 6/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 39s 49ms/step - accuracy: 0.9789 - loss: 0.0574 - val_accuracy: 0.9894 - val_loss: 0.0381
Epoch 7/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 38s 48ms/step - accuracy: 0.9794 - loss: 0.0565 - val_accuracy: 0.9875 - val_loss: 0.0406
Epoch 8/10
788/788 ━━━━━━━━━━━━━━━━━━━━ 40s 50ms/step - accuracy: 0.9809 - loss: 0.0535 - 

In [ ]:
# ===== 1) قارن بالنموذج الأصلي (قبل adversarial training) =====
# النموذج اللي اتدرب على clean data فقط
loss_clean, acc_clean_clean = ids_numeric.evaluate(X_test_numeric_lstm, y_test_numeric_lstm)
print(f"Original Model (before adversarial training):")
print(f"  Clean Accuracy: {acc_clean_clean:.4f}")

# اختبره على adversarial examples
loss_clean_adv, acc_clean_adv = ids_numeric.evaluate(X_adv_numeric_lstm, y_adv_numeric_lstm)
print(f"  Adversarial Accuracy: {acc_clean_adv:.4f}")
print(f"  Drop: {acc_clean_clean - acc_clean_adv:.4f}")

print("\n" + "="*60)

# ===== 2) النموذج بعد adversarial training =====
print(f"Robust Model (after adversarial training):")
print(f"  Clean Accuracy: {acc_adv_clean:.4f}")
print(f"  Adversarial Accuracy:  {acc_adv_adv:.4f}")
print(f"  Improvement: {acc_adv_adv - acc_clean_adv:.4f}")

705/705 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7542 - loss: 1.0318
Original Model (before adversarial training):
  Clean Accuracy: 0.7530
705/705 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7189 - loss: 1.4286
  Adversarial Accuracy: 0.7161
  Drop: 0.0370

Robust Model (after adversarial training):
  Clean Accuracy: 0.7462
  Adversarial Accuracy: 0.7471
  Improvement: 0.0310


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow as tf

def build_tuned_lstm(input_dim,
                     lstm_units=[64, 32],
                     dropout_rate=0.3,
                     bidirectional=False,
                     batch_norm=False,
                     learning_rate=0.001,
                     activation='tanh'):
    """Build LSTM model using tunable architecture and activation."""

    model = Sequential()

    # First LSTM layer
    if bidirectional:
        model.add(Bidirectional(
            LSTM(
                lstm_units[0],
                activation=activation,
                return_sequences=True if len(lstm_units) > 1 else False
            ),
            input_shape=(1, input_dim)
        ))
    else:
        model.add(LSTM(
            lstm_units[0],
            activation=activation,
            return_sequences=True if len(lstm_units) > 1 else False,
            input_shape=(1, input_dim)
        ))

    if batch_norm:
        model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # Additional LSTM layers
    for i, units in enumerate(lstm_units[1:], 1):
        if bidirectional:
            model.add(Bidirectional(
                LSTM(
                    units,
                    activation=activation,
                    return_sequences=i < len(lstm_units) - 1
                )
            ))
        else:
            model.add(LSTM(
                units,
                activation=activation,
                return_sequences=i < len(lstm_units) - 1
            ))

        if batch_norm:
            model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model



In [ ]:
# ============================================================
#  DHOLE للـ Parameter Tuning (بدل Feature Selection)
# ============================================================

# ===== 1) Parameter Space =====
num_numeric_features = len(numeric_idx)
param_configs = {
    'n_layers': {'type': 'categorical', 'values': [1, 2, 3]},
    'units_layer1': {'type': 'categorical', 'values': [64, 128, 256]},
    'units_layer2': {'type': 'categorical', 'values': [0, 32, 64, 128]},
    'units_layer3': {'type': 'categorical', 'values': [0, 32, 64]},
    'dropout': {'type': 'float', 'min': 0.2, 'max': 0.5},
    'bidirectional': {'type': 'categorical', 'values': [0, 1]},
    'learning_rate': {'type': 'float', 'min': 0.0001, 'max': 0.002},
    'batch_size': {'type': 'categorical', 'values': [128, 256, 512]},
    'batch_norm': {'type': 'categorical', 'values': [0, 1]},
    'activation': {'type': 'categorical', 'values': ['relu', 'tanh', 'sigmoid']}
}

# حساب dimension
total_dim = 0
for config in param_configs.values():
    if config['type'] == 'categorical':
        total_dim += len(config['values'])
    elif config['type'] == 'integer':
        total_dim += 4
    elif config['type'] == 'float':
        total_dim += 8

print(f"Total binary dimension: {total_dim}")


def _safe_argmax(bits, segment_offset=0):
    """Avoid bias to index 0 when all bits are zero."""
    "\"\"Deterministically pick an index, even when all bits are zero.\"\"\"\n"
    if np.sum(bits) == 0:
        #return np.random.randint(len(bits))
        return int(segment_offset % len(bits))
    return int(np.argmax(bits))


def decode_binary_to_params(binary, param_configs):
    """تحويل binary encoding لـ parameters"""
    params = {}
    idx = 0

    for param_name, config in param_configs.items():
        if config['type'] == 'categorical':
            n_values = len(config['values'])
            one_hot = binary[idx:idx+n_values]
            #selected_idx = _safe_argmax(one_hot)
            selected_idx = _safe_argmax(one_hot, segment_offset=idx)
            params[param_name] = config['values'][selected_idx]
            idx += n_values

        elif config['type'] == 'integer':
            bits = binary[idx:idx+4]
            normalized = int(''.join(str(int(b)) for b in bits), 2)
            value = int(config['min'] + (normalized / 15) * (config['max'] - config['min']))
            params[param_name] = value
            idx += 4

        elif config['type'] == 'float':
            bits = binary[idx:idx+8]
            normalized = int(''.join(str(int(b)) for b in bits), 2)
            value = config['min'] + (normalized / 255) * (config['max'] - config['min'])
            params[param_name] = value
            idx += 8

    return params


def _params_key(params):
    """Create hashable cache key with rounded floats."""
    key = []
    for k, v in params.items():
        if isinstance(v, float):
            key.append((k, round(v, 6)))
        else:
            key.append((k, v))
    return tuple(key)


fitness_cache = {}


def fitness_function(binary_mask):
    """Fitness for robust accuracy on clean+adversarial."""
    params = decode_binary_to_params(binary_mask, param_configs)
    key = _params_key(params)
    if key in fitness_cache:
        return fitness_cache[key]

    try:
        lstm_units = []
        if params['n_layers'] >= 1:
            lstm_units.append(params['units_layer1'])
        if params['n_layers'] >= 2 and params['units_layer2'] > 0:
            lstm_units.append(params['units_layer2'])
        if params['n_layers'] >= 3 and params['units_layer3'] > 0:
            lstm_units.append(params['units_layer3'])

        if len(lstm_units) == 0:
            return np.inf

        tf.keras.backend.clear_session()
        model = build_tuned_lstm(
            input_dim=num_numeric_features,
            lstm_units=lstm_units,
            dropout_rate=params['dropout'],
            bidirectional=bool(params['bidirectional']),
            batch_norm=bool(params['batch_norm']),
            learning_rate=params['learning_rate'],
            activation=params['activation']
        )

        history = model.fit(
            X_train_combined_lstm,
            y_train_combined_lstm,
            epochs=15,
            batch_size=int(params['batch_size']),
            validation_split=0.2,
            callbacks=[EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
            verbose=0
        )

        _, acc_clean = model.evaluate(X_test_numeric_lstm, y_test_numeric_lstm, verbose=0)
        _, acc_adv = model.evaluate(X_adv_numeric_lstm, y_adv_numeric_lstm, verbose=0)

        avg_acc = (acc_clean + acc_adv) / 2
        gap = abs(acc_clean - acc_adv)
        val_loss_last = history.history['val_loss'][-1]

        fitness = -(avg_acc - 0.15 * gap - 0.02 * val_loss_last)

        print(f"  Clean: {acc_clean:.3f}, Adv: {acc_adv:.3f}, Avg: {avg_acc:.3f}")
        fitness_cache[key] = fitness
        return fitness

    except Exception as e:
        print(f"  Error: {e}")
        return np.inf


def dhole_opt(N=10, T=10):
    """DHOLE optimization"""
    dim = total_dim
    X = np.random.randint(0, 2, size=(N, dim))

    print('')
    print('=' * 60)
    print(f"Starting DHOLE: N={N}, T={T}, dim={dim}")
    print('=' * 60)

    fitness = np.zeros(N)
    for i in range(N):
        print(f"Init {i+1}/{N}:")
        fitness[i] = fitness_function(X[i])

    best_idx = np.argmin(fitness)
    prey_global = X[best_idx].copy()
    best_fit = fitness[best_idx]

    curve = []

    print('')
    print('=' * 60)
    print(f"Initial Best: {-best_fit:.4f}")
    print('=' * 60)

    for t in range(T):
        C = 1 - t / T
        print('')
        print(f"--- Iteration {t+1}/{T} ---")

        for i in range(N):
            Xnew = X[i].copy()

            if np.random.rand() < 0.5:
                n_flips = max(1, int(dim * 0.1))
                flip_indices = np.random.choice(dim, n_flips, replace=False)
                Xnew[flip_indices] = 1 - Xnew[flip_indices]
            else:
                diff = prey_global ^ X[i]
                flip = np.random.rand(dim) < (C * diff)
                Xnew[flip] = prey_global[flip]

            new_fit = fitness_function(Xnew)

            if new_fit < fitness[i]:
                X[i] = Xnew
                fitness[i] = new_fit

                if new_fit < best_fit:
                    best_fit = new_fit
                    prey_global = Xnew.copy()
                    print(f"  → NEW BEST: {-best_fit:.4f}")

        curve.append(-best_fit)
        print(f"Best: {-best_fit:.4f}")

    best_params = decode_binary_to_params(prey_global, param_configs)
    return best_params, -best_fit, curve


# ===== 5) Run =====
best_params, best_score, curve = dhole_opt(N=10, T=12)

print('')
print('=' * 60)
print('BEST PARAMETERS:')
print('=' * 60)
for k, v in best_params.items():
    print(f"  {k}: {v}")
print('')
print(f"Best Score: {best_score:.4f}")

# ===== 6) Train Final Model =====
print('')
print('=' * 60)
print('Training FINAL model...')
print('=' * 60)

lstm_units_final = []
if best_params['n_layers'] >= 1:
    lstm_units_final.append(best_params['units_layer1'])
if best_params['n_layers'] >= 2 and best_params['units_layer2'] > 0:
    lstm_units_final.append(best_params['units_layer2'])
if best_params['n_layers'] >= 3 and best_params['units_layer3'] > 0:
    lstm_units_final.append(best_params['units_layer3'])

final_model = build_tuned_lstm(
    input_dim=num_numeric_features,
    lstm_units=lstm_units_final,
    dropout_rate=best_params['dropout'],
    bidirectional=bool(best_params['bidirectional']),
    batch_norm=bool(best_params['batch_norm']),
    learning_rate=best_params['learning_rate'],
    activation=best_params['activation']
)

final_model.fit(
    X_train_combined_lstm,
    y_train_combined_lstm,
    epochs=40,
    batch_size=int(best_params['batch_size']),
    validation_split=0.2,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ],
    verbose=1
)

# ===== 7) Evaluate =====
_, acc_clean = final_model.evaluate(X_test_numeric_lstm, y_test_numeric_lstm)
_, acc_adv = final_model.evaluate(X_adv_numeric_lstm, y_adv_numeric_lstm)

print('')
print('=' * 60)
print('FINAL RESULTS:')
print('=' * 60)
print(f"Clean Accuracy:       {acc_clean:.4f}")
print(f"Adversarial Accuracy: {acc_adv:.4f}")
print(f"Gap:                  {abs(acc_clean - acc_adv):.4f}")
print(f"Average:              {(acc_clean + acc_adv)/2:.4f}")
print('=' * 60)



Total binary dimension: 39

Starting DHOLE: N=10, T=12, dim=39

Init 1/10:
  Clean: 0.805, Adv: 0.795, Avg: 0.800
Init 2/10:
  Clean: 0.759, Adv: 0.765, Avg: 0.762
Init 3/10:
  Clean: 0.778, Adv: 0.778, Avg: 0.778
Init 4/10:
  Clean: 0.774, Adv: 0.769, Avg: 0.771
Init 5/10:
  Clean: 0.793, Adv: 0.783, Avg: 0.788
Init 6/10:
  Clean: 0.763, Adv: 0.762, Avg: 0.763
Init 7/10:
  Clean: 0.756, Adv: 0.749, Avg: 0.752
Init 8/10:
  Clean: 0.786, Adv: 0.769, Avg: 0.777
Init 9/10:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Clean: 0.744, Adv: 0.743, Avg: 0.743
Init 10/10:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Clean: 0.759, Adv: 0.746, Avg: 0.752

Initial Best: 0.7990


--- Iteration 1/12 ---
  Clean: 0.783, Adv: 0.778, Avg: 0.781


In [ ]:
# Selected features by Dhole


def fitness_function(mask):
    """
    mask: binary vector (0/1) length = number of features
    """
    # avoid empty feature set
    if np.sum(mask) == 0:
        return np.inf

    selected_idx = np.where(mask == 1)[0]

    X_sel = X_train.iloc[:, selected_idx]
    X_val_sel = X_test.iloc[:, selected_idx]

    # simple IDS model (fast evaluation)
    model = LogisticRegression(max_iter=2)
    model.fit(X_sel, y_train)

    y_pred = model.predict(X_val_sel)
    acc = accuracy_score(y_test, y_pred)

    # penalty for using too many features
    penalty = 0.01 * np.sum(mask)

    return -(acc - penalty)   # minimization

def dhole_opt(N=20, T=5):
    dim = X_train.shape[1]

    # initialize population (binary masks)
    X = np.random.randint(0, 2, size=(N, dim))

    fitness = np.array([fitness_function(ind) for ind in X])

    best_idx = np.argmin(fitness)
    prey_global = X[best_idx].copy()
    best_fit = fitness[best_idx]

    curve = []

    for t in range(T):
        C = 1 - t / T

        for i in range(N):
            Xnew = X[i].copy()

            if np.random.rand() < 0.5:
                # exploration
                j = np.random.randint(dim)
                Xnew[j] = 1 - Xnew[j]
            else:
                # exploitation (move toward best)
                diff = prey_global ^ X[i]
                flip = np.random.rand(dim) < (C * diff)
                Xnew[flip] = prey_global[flip]

            new_fit = fitness_function(Xnew)

            if new_fit < fitness[i]:
                X[i] = Xnew
                fitness[i] = new_fit

                if new_fit < best_fit:
                    best_fit = new_fit
                    prey_global = Xnew.copy()

        curve.append(best_fit)

    return prey_global, best_fit, curve

best_mask, best_score, curve = dhole_opt()

selected_features = X_train.columns[best_mask == 1]

print("Selected features:", list(selected_features))
print("Number of features:", len(selected_features))

#test LSTM accuracy after selection features
model_opt = build_ids_lstm(len(selected_features))
X_train_sel = X_train.loc[:, selected_features]
X_test_sel  = X_test.loc[:, selected_features]
X_train_sel_lstm = np.expand_dims(X_train_sel.values, axis=1)
X_test_sel_lstm  = np.expand_dims(X_test_sel.values, axis=1)

model_opt.fit(
    X_train_sel_lstm,
    y_train,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

opt_acc = model_opt.evaluate(
          X_test_sel_lstm, y_test # Corrected indexing
)[1]
print("Test shape:", X_test_sel_lstm.shape)
print("Train shape:", X_train_sel_lstm.shape)
print("Optimized Accuracy:", opt_acc)

# predictions
y_pred_prob = model_opt.predict(X_test_sel_lstm).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

print(" LSTM Accuracy:", accuracy_score(y_test, y_pred))
print(" LSTM AUC:", roc_auc_score(y_test, y_pred_prob))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Selected features: ['protocol_type', 'flag', 'num_root', 'srv_count', 'dst_host_count', 'dst_host_srv_count', 'dst_host_diff_srv_rate', 'dst_host_srv_diff_host_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']
Number of features: 10
Epoch 1/50
443/443 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9002 - loss: 0.2550 - val_accuracy: 0.9524 - val_loss: 0.1433
Epoch 2/50
443/443 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9502 - loss: 0.1454 - val_accuracy: 0.9632 - val_loss: 0.1144
Epoch 3/50
443/443 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9566 - loss: 0.1280 - val_accuracy: 0.9675 - val_loss: 0.0943
Epoch 4/50
443/443 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9609 - loss: 0.1169 - val_accuracy: 0.9694 - val_loss: 0.0900
Epoch 5/50
443/443 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9639 - loss: 0.1066 - val_accuracy: 0.9702 - val_loss: 0.0839
Epoch 6/50
443/443 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9654 - loss: 0.1023 - val_accuracy: 0.9711 - 

In [ ]:
# ===============================
#   PART 3 —  an other way to Generate Adversarial Examples (FGSM) using problem space
# ===============================
# ===============================
# FGSM EXACTLY
# ===============================
# -------- 1) Problem-space numeric features only --------
categorical_features = ['protocol_type','service','flag']
# #attack on all weak features
# numeric_features = [
#     f for f in X_train.columns
#     if f not in selected_features and f not in categorical_features
# ]
#attack on all strong features
numeric_features = [
    f for f in selected_features
    if f not in categorical_features
]
# -------- 2) All categorical that must NOT change --------
# copy of data before encoding
# raw_test = pd.read_csv(test_url, names=cols)
# raw_test = raw_test.drop(columns=['difficulty_score'])
# raw_test['label'] = raw_test['label'].apply(lambda x: 0 if x == 'normal' else 1)
# X_raw_41 = raw_test.drop('label', axis=1)
# X_encoded_121 = pd.get_dummies(X_raw_41, columns=categorical_features)
# X_train_encoded_cols = X_encoded_121.columns

def manda_fgsm(model, X_raw_unencoded_full, X_encoded_aligned_for_model, y_labels, eps=0.1, full_model_cols=None, subset_model_cols=None, p=0.05):

    # ---------- Step 1: FGSM in feature-space (on subset features) ----------
    X_model_input_tensor = tf.convert_to_tensor(X_encoded_aligned_for_model, dtype=tf.float32)
    X_model_input_tensor = tf.expand_dims(X_model_input_tensor, axis=1) # Add the timesteps dimension for LSTM
    y_tensor = tf.convert_to_tensor(y_labels, dtype=tf.float32)
    y_tensor = tf.expand_dims(y_tensor, axis=-1)

    with tf.GradientTape() as tape:
        tape.watch(X_model_input_tensor)
        pred = model(X_model_input_tensor)
        loss = tf.keras.losses.binary_crossentropy(y_tensor, pred)

    grad = tape.gradient(loss, X_model_input_tensor).numpy()
    grad_sign = np.sign(grad) # grad_sign will have shape (batch_size, 1, len(subset_model_cols))

    if full_model_cols is None or subset_model_cols is None:
        raise ValueError("full_model_cols and subset_model_cols must be provided for problem-space FGSM")

    # Nullify perturbations on categorical / non-diff features within the subset
    for cat in categorical_features:
        # Find indices of one-hot encoded columns corresponding to the categorical feature within subset_model_cols
        col_indices_in_subset = [i for i, c in enumerate(subset_model_cols) if c.startswith(cat+'_')]
        if col_indices_in_subset:
            grad_sign[:, 0, col_indices_in_subset] = 0 # This now correctly indexes into the subset grad_sign

    # ---------- Step 2: Map back to problem-space ----------
    X_raw_adv = X_raw_unencoded_full.copy() # Operate on the full raw DataFrame

    for f in numeric_features:
        # Check if the numeric feature exists in the subset_model_cols (because grad_sign is based on it)
        if f in subset_model_cols:
            idx_in_subset = subset_model_cols.get_loc(f)
            g = grad_sign[:, 0, idx_in_subset] # FGSM direction (+1 / -1)
            #g = np.sign(grad_sign[:, 0, :].mean(axis=1))
            delta = p * X_raw_adv[f].abs()     # allowed change = p%

            # apply modification
            X_raw_adv[f] = X_raw_adv[f] +  (eps * g * delta)
            X_raw_adv[f] = np.clip(X_raw_adv[f], 0, None) # Ensure non-negative

    # Re-encode categorical features from the modified raw data
    adv = pd.get_dummies(X_raw_adv, columns=categorical_features)

    # Align with the full model's expected columns (X_train.columns) for the final output
    X_adv_encoded = adv.reindex(columns=full_model_cols, fill_value=0)

    # Separate X and y
    X_final = X_adv_encoded.astype(float).values
    y_final = y_labels

    return X_final, y_final


# # # Generate 5k adversarial samples (using 200 samples for consistency)
X_raw_subset_full = X_train.reset_index(drop=True)
# Filter X_test for selected features and consistent row count
X_test_encoded_for_model = X_train.loc[:, selected_features].values # Corrected: Use selected_features
y_subset = y_train.values

# Diagnostic prints to verify shapes before calling manda_fgsm
print(f"Shape of X_raw_subset_full: {X_raw_subset_full.shape}")
print(f"Shape of X_test_encoded_for_model: {X_test_encoded_for_model.shape}")
print(f"Shape of y_subset: {y_subset.shape}")
X_adv, y_adv = manda_fgsm(
    model_opt,
    X_raw_subset_full,
    X_test_encoded_for_model,
    y_subset,
    eps=0.1,
    full_model_cols=X_train.columns,
    subset_model_cols=pd.Index(selected_features), # Corrected: Convert selected_features to a Pandas Index
    p=0.075
)

print("Generated adversarial examples (problem space):", X_adv.shape)

#show which features affected by FGSM
diff = pd.DataFrame(X_adv, columns=X_train.columns) - X_test # Convert X_adv to DataFrame with correct columns for comparison
changed_features = (diff[numeric_features] != 0).sum(axis=0)
print(changed_features)


Shape of X_raw_subset_full: (125973, 41)
Shape of X_test_encoded_for_model: (125973, 10)
Shape of y_subset: (125973,)
Generated adversarial examples (problem space): (125973, 41)
num_root                       103601
srv_count                      125968
dst_host_count                 125969
dst_host_srv_count             125973
dst_host_diff_srv_rate         122588
dst_host_srv_diff_host_rate    114725
dst_host_rerror_rate           115044
dst_host_srv_rerror_rate       113047
dtype: int64


In [ ]:
# adversarial examples generated on strong features
#then test on clean & adversarial strong**

#when use all best features
#selected_idx = [X_train.columns.get_loc(f) for f in selected_features]
# when use best features (selected + non-categorical)
strong_idx = [
    X_train.columns.get_loc(f)
    for f in selected_features
    if f not in categorical_features
]
# clean
X_train_sel = X_train.values[:, strong_idx]
X_test_sel  = X_test.values[:, strong_idx]

# adversarial
X_adv_sel = X_adv[:, strong_idx]


#merge training data and attack data
X_train_adv = np.vstack([X_train_sel, X_adv_sel])
y_train_adv = np.concatenate([y_train.values, y_adv])

# shuffle
idx = np.random.permutation(len(X_train_adv))
X_train_adv = X_train_adv[idx]
y_train_adv = y_train_adv[idx]

# change dimension to time step 5
X_train_adv_lstm, y_train_adv_lstm = make_lstm_sequences(
    X_train_adv, y_train_adv, timesteps=5
)

X_test_lstm, y_test_lstm = make_lstm_sequences(
    X_test_sel, y_test.values, timesteps=5
)

X_adv_sel_lstm, y_adv_sel_lstm = make_lstm_sequences(
    X_adv_sel, y_adv, timesteps=5
)

fgsm_selected = build_ids_lstm(len(strong_idx))

##############################################type one #############################
#train LSTM  on clean training data+ adversarial
# Train: Clean+adversarial
# Test : Adversarial
# Train on CLEAN and adversarial (Strong features)
history = fgsm_selected.fit(
    X_train_adv_lstm,
    y_train_adv_lstm ,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

# Test on clean
loss_clean, acc_clean = fgsm_selected.evaluate(X_test_lstm, y_test_lstm)
print("Accuracy on clean test:", acc_clean)

# Test on adversarial
loss_adv, acc_adv = fgsm_selected.evaluate(X_adv_sel_lstm, y_adv_sel_lstm)
print("Accuracy on adversarial test:", acc_adv)



############################################## type two #############################
#train LSTM  on clean training data
# Train: Clean
# Test : Adversarial

#add lstm dimesion in 5 time step for training
# X_train_num_lstm, y_train_num_lstm = make_lstm_sequences(
#     X_train_sel, y_train, timesteps=timesteps_for_lstm
# )

# fgsmselclean= build_ids_lstm(len(strong_idx))

# history = fgsmselclean.fit(
#     X_train_num_lstm,
#     Y_train_num_lstm,
#     epochs=50,
#     batch_size=256,
#     validation_split=0.2,
#     verbose=1
# )

# # Test on clean
# loss_clean, acc_clean = fgsmselclean.evaluate(X_test_lstm, y_test_lstm)
# print("Accuracy on clean test:", acc_clean)

# # Test on adversarial
# loss_adv, acc_adv = fgsmselclean.evaluate(X_adv_sel_lstm, y_adv_sel_lstm)
# print("Accuracy on adversarial test:", acc_adv)



Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


326/788 ━━━━━━━━━━━━━━━━━━━━ 18s 39ms/step - accuracy: 0.8480 - loss: 0.3581

KeyboardInterrupt: 

In [ ]:
# ===============================
#   PART 4 — Compute MANIFOLD SCORE
# ===============================

# PCA projection distance as manifold score
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

pca = PCA(n_components=10)
pca.fit(X_train)

def manifold_score(x):
    proj = pca.inverse_transform(pca.transform(x))
    return np.mean((x - proj)**2, axis=1)

manifold_clean = manifold_score(X_train[:])
manifold_adv   = manifold_score(X_adv)

In [ ]:
# ===============================
#   PART 5 — Compute DB SCORE (Deep Boundary)
# ===============================

def db_score(model, x):
    # Reshape input for the LSTM model
    x_reshaped = np.expand_dims(x, axis=1)
    with tf.GradientTape() as tape:
        x_t = tf.convert_to_tensor(x_reshaped, dtype=tf.float32)
        tape.watch(x_t)
        pred = model(x_t)
    grad = tape.gradient(pred, x_t).numpy()
    # The gradient will also be 3D (batch, 1, features), so squeeze the middle dimension
    return np.mean(np.abs(grad[:, 0, :]), axis=1)

db_clean = db_score(ids, X_train[:])
db_adv   = db_score(ids, X_adv)



In [ ]:
# ===============================
#   PART 6 — Build MANDA dataset
# ===============================

S1 = np.concatenate([manifold_clean, manifold_adv])
S2 = np.concatenate([db_clean, db_adv])
Y  = np.concatenate([np.zeros_like(manifold_clean), np.ones_like(manifold_adv)])

df_manda = pd.DataFrame({'manifold': S1, 'db': S2, 'label': Y})
df_manda.head()
#print(df_manda)


In [ ]:
# ===============================
#   PART 7 — Train MANDA (Logistic Regression)
# ===============================

# clf = LogisticRegression()
# clf.fit(df_manda[['manifold', 'db']], df_manda['label'])

# pred = clf.predict(df_manda[['manifold', 'db']])
# print("MANDA accuracy:", accuracy_score(df_manda['label'], pred))
# print("AUC:", roc_auc_score(df_manda['label'], pred))

# ===============================
#   PART 7 — Train MANDA (LSTM)
# ===============================
# MANDA features
X_manda = df_manda[['manifold', 'db']].values
y_manda = df_manda['label'].values

# reshape for LSTM: (samples, timesteps=1, features=2)
X_manda_lstm = X_manda.reshape(X_manda.shape[0], 1, X_manda.shape[1])
manda_lstm=build_ids_lstm(X_manda_lstm.shape[2]) # Corrected: Use shape[2] for number of features
manda_lstm.fit(
    X_manda_lstm,
    y_manda,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

# predictions
y_pred_prob = manda_lstm.predict(X_manda_lstm).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

print("MANDA LSTM Accuracy:", accuracy_score(y_manda, y_pred))
print("MANDA LSTM AUC:", roc_auc_score(y_manda, y_pred_prob))


In [ ]:
# ===============================
#   PART 8 — Fixing FPR at 5% or 15%
# ===============================
manda_features = ['manifold', 'db']

X_manda = df_manda[manda_features].values
y_manda = df_manda['label'].values

# convert to LSTM sequences
X_manda_lstm, y_manda_lstm = make_lstm_sequences(
    X_manda, y_manda, timesteps=5
)
scores = manda_lstm.predict(X_manda_lstm).ravel()
fpr, tpr, th = roc_curve(y_manda_lstm, scores)

def get_threshold(target_fpr):
    idx = np.argmin(np.abs(fpr - target_fpr))
    return th[idx]

thr_5  = get_threshold(0.05)
thr_15 = get_threshold(0.15)

print("Threshold at 5% FPR:", thr_5)
print("Threshold at 15% FPR:", thr_15)
